# Forêts aléatoires : dataset [AMES](http://jse.amstat.org/v19n3/decock.pdf)

## Introduction

Dans ce TP, nous prédisons le prix de vente des maisons d'Ames (Iowa) avec une forêt aléatoire, sur les données que le TP d'exploration (`ml/ames-data-exploration`) a préparées. L'objectif est de voir ce que la forêt apporte par rapport à un arbre seul, et ce qu'elle donne en plus d'une prédiction.

Nous allons suivre ces étapes :

1. écrire une fonction d'évaluation par validation croisée
2. comparer un arbre seul et une forêt
3. régler les hyperparamètres de la forêt
4. lire l'importance par impureté des variables, et le premier arbre de la forêt
5. suivre l'erreur out-of-bag (OOB) selon le nombre d'arbres
6. évaluer le modèle retenu sur un jeu de validation mis de côté dès le début

## Données

Les fichiers de `ames/processed/` du dépôt [shuuchuu/datasets](https://github.com/shuuchuu/datasets/tree/main/ames) sont les données du TP d'exploration, préparées comme dans sa solution : valeurs manquantes remplies, variables ordinales codées par des entiers, variables catégorielles encodées en one-hot, soit 244 colonnes pour 1 460 maisons. `train_labels.csv` contient la cible, `SalePrice`.

Les conclusions de la fin de l'exploration (retirer les 4 ventes de plus de 4 000 sq ft, prédire le logarithme du prix) ne sont pas appliquées ici : un arbre ne suppose rien de la distribution de la cible, et l'erreur reste ainsi en dollars.

On met de côté dès maintenant 20 % des maisons, un jeu de validation qui ne servira à aucun choix et qu'on n'utilisera qu'à la fin du TP. Tout le reste se fait sur `train_X` et `train_y`.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ames
!ls -l ames/processed/

In [ ]:
import matplotlib.pyplot as plt
import pandas
import scipy.stats
import sklearn.base
import sklearn.ensemble
import sklearn.metrics
import sklearn.model_selection
import sklearn.tree

X = pandas.read_csv("ames/processed/train_data.csv", index_col="Id")
y = pandas.read_csv("ames/processed/train_labels.csv", index_col="Id")["SalePrice"]

# Jeu de validation : 20 % des maisons, mises de côté jusqu'à la fin du TP
train_X, val_X, train_y, val_y = sklearn.model_selection.train_test_split(
  X, y, test_size=0.2, random_state=0
)
print(f"Entraînement : {train_X.shape}, validation : {val_X.shape}")

## Évaluation

*Écrivez une fonction `score` qui évalue un modèle par validation croisée à 5 plis sur `train_X` et `train_y`, avec la RMSE (racine de l'erreur quadratique moyenne), et renvoie la moyenne et l'écart-type de la RMSE sur les 5 plis.*

Dans scikit-learn, un score est d'autant meilleur qu'il est grand : la RMSE s'y utilise donc par son opposé, le score `"neg_root_mean_squared_error"`.

**Documentation utile**

- [`sklearn.model_selection.cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [Les scores de scikit-learn](https://scikit-learn.org/stable/modules/model_evaluation.html#string-name-scorers)

In [ ]:
def score(model):
  # Votre code ici
  ...

### Conseils supplémentaires

- `sklearn.model_selection.cross_val_score(model, train_X, train_y, cv=5, scoring="neg_root_mean_squared_error")` : un tableau de 5 scores, un par pli, qui sont des RMSE négatives.
- `rmse.mean()` et `rmse.std()` : la moyenne et l'écart-type d'un tableau NumPy.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def score(model):
  rmse = -sklearn.model_selection.cross_val_score(
    model, train_X, train_y, cv=..., scoring=...
  )
  return ..., ...
```

### Solution

In [ ]:
def score(model):
  rmse = -sklearn.model_selection.cross_val_score(
    model, train_X, train_y, cv=5, scoring="neg_root_mean_squared_error"
  )
  return rmse.mean(), rmse.std()

## Un arbre ou une forêt

Un arbre de décision profond surapprend : il varie beaucoup d'un échantillon d'entraînement à l'autre. La forêt moyenne des arbres décorrélés pour réduire cette variance.

*Évaluez avec `score` un `DecisionTreeRegressor` seul, puis une `RandomForestRegressor` de 100 arbres. Lequel se trompe le moins en moyenne ? Lequel varie le moins d'un pli à l'autre ?*

**Documentation utile**

- [`sklearn.tree.DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
- [`sklearn.ensemble.RandomForestRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `random_state=0` dans les deux modèles : des résultats identiques d'une exécution à l'autre.
- `n_estimators=100` : le nombre d'arbres de la forêt ; `n_jobs=-1` les entraîne en parallèle sur tous les cœurs.
- `score` renvoie un tuple `(moyenne, écart-type)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
tree_rmse = score(sklearn.tree.DecisionTreeRegressor(random_state=0))
forest_rmse = score(...)
print(f"Arbre seul : {tree_rmse[0]:.0f} $ ± {tree_rmse[1]:.0f} $")
print(f"Forêt de 100 arbres : ...")
```

### Solution

In [ ]:
tree_rmse = score(sklearn.tree.DecisionTreeRegressor(random_state=0))
forest_rmse = score(
  sklearn.ensemble.RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
)
print(f"Arbre seul : {tree_rmse[0]:.0f} $ ± {tree_rmse[1]:.0f} $")
print(f"Forêt de 100 arbres : {forest_rmse[0]:.0f} $ ± {forest_rmse[1]:.0f} $")

L'arbre seul se trompe d'environ 41 300 $ en moyenne, la forêt de 100 arbres d'environ 28 900 $ : près d'un tiers d'erreur en moins. L'écart-type, lui, ne baisse pas (environ 5 500 $ pour l'arbre, 6 200 $ pour la forêt) : il mêle la variance du modèle, que la moyenne d'arbres réduit, et les différences entre les plis (certains contiennent des maisons plus difficiles), que rien ne réduit. Ici, ce sont les différences entre plis qui dominent.

## Optimisation

*Utilisez `RandomizedSearchCV` pour régler les paramètres `n_estimators`, `max_features`, `max_depth`, `min_samples_leaf` et `max_samples` de la forêt, avec la même validation que `score` (5 plis, RMSE). La meilleure configuration fait-elle mieux que la forêt de 100 arbres de la section précédente ?*

`RandomizedSearchCV` tire au hasard `n_iter` configurations. Pour chaque paramètre, elle prend une liste de valeurs ou, surtout, une loi de probabilité de `scipy.stats`, qui explore toute une plage : c'est ce qui la distingue de `GridSearchCV`, qui essaie toutes les combinaisons d'une grille. La recherche entraîne 100 forêts (20 configurations, 5 plis) : comptez quelques minutes.

**Documentation utile**

- [`sklearn.model_selection.RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
- [`scipy.stats.randint`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.randint.html)
- [`scipy.stats.uniform`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.uniform.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Des plages raisonnables pour `param_distributions` :

- `n_estimators=scipy.stats.randint(100, 300)` : un entier de 100 à 299 ; plus d'arbres ne surapprend pas, mais coûte du temps.
- `max_features=["sqrt", "log2", 0.33, None]` : le nombre de variables tirées à chaque séparation ; un flottant en donne la fraction (0.33 : le c/3 du paquet R `randomForest`), `None` les prend toutes (le défaut de scikit-learn en régression).
- `max_depth=[8, 16, None]` : `None` laisse pousser les arbres jusqu'au bout.
- `min_samples_leaf=scipy.stats.randint(1, 6)` : de 1 à 5 exemples par feuille au minimum.
- `max_samples=scipy.stats.uniform(0.5, 0.5)` : la fraction des lignes tirées pour chaque arbre, entre 0,5 et 1 (`uniform(loc, scale)` tire dans l'intervalle [loc, loc + scale]).

Et pour la recherche : `n_iter=20`, `cv=5`, `scoring="neg_root_mean_squared_error"`, `random_state=0` (dans la recherche et dans la forêt), `n_jobs=-1`. Après `fit`, `rscv.best_score_` est la meilleure RMSE moyenne, en négatif, `rscv.best_params_` sa configuration, et `rscv.best_estimator_` la forêt réentraînée avec elle sur tout `train_X`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rscv = sklearn.model_selection.RandomizedSearchCV(
  estimator=sklearn.ensemble.RandomForestRegressor(random_state=0),
  param_distributions=dict(
    n_estimators=scipy.stats.randint(100, 300),
    max_features=["sqrt", "log2", 0.33, None],
    max_depth=...,
    min_samples_leaf=...,
    max_samples=...,
  ),
  n_iter=20,
  scoring=...,
  cv=...,
  random_state=0,
  n_jobs=-1,
)
rscv.fit(...)

print(f"Meilleure RMSE en validation croisée : {-rscv.best_score_:.0f} $")
print(f"Meilleurs paramètres : {rscv.best_params_}")
rfr = rscv.best_estimator_
```

### Solution

In [ ]:
rscv = sklearn.model_selection.RandomizedSearchCV(
  estimator=sklearn.ensemble.RandomForestRegressor(random_state=0),
  param_distributions=dict(
    n_estimators=scipy.stats.randint(100, 300),
    max_features=["sqrt", "log2", 0.33, None],
    max_depth=[8, 16, None],
    min_samples_leaf=scipy.stats.randint(1, 6),
    max_samples=scipy.stats.uniform(0.5, 0.5),
  ),
  n_iter=20,
  scoring="neg_root_mean_squared_error",
  cv=5,
  random_state=0,
  n_jobs=-1,
)
rscv.fit(train_X, train_y)

print(f"Meilleure RMSE en validation croisée : {-rscv.best_score_:.0f} $")
print(f"Forêt de 100 arbres par défaut : {forest_rmse[0]:.0f} $")
print(f"Meilleurs paramètres : {rscv.best_params_}")

# Meilleur modèle, réentraîné sur tout train_X
rfr = rscv.best_estimator_

La meilleure configuration (profondeur 16, un tiers des variables à chaque séparation, 61 % des lignes par arbre, 2 exemples par feuille au minimum, 179 arbres) atteint environ 28 300 $, contre 28 900 $ pour la forêt par défaut : à peine 2 % de mieux. Une forêt aléatoire marche bien avec ses réglages par défaut ; la recherche sert surtout à vérifier qu'aucun réglage ne change beaucoup le résultat.

## Importance des variables

Une forêt de centaines d'arbres ne se lit plus comme un arbre seul : on l'interprète par l'importance de ses variables. L'importance par impureté (`feature_importances_`) d'une variable est la baisse d'impureté (ici, de l'erreur quadratique) qu'apportent les séparations sur cette variable, en moyenne sur les arbres. Elle est rapide à obtenir, mais calculée sur les données d'entraînement, et biaisée vers les variables à beaucoup de valeurs.

*Tracez l'importance par impureté des 20 variables les plus importantes de `rfr`. Retrouvez-vous les variables que l'exploration trouvait les plus corrélées au prix, la qualité générale (`OverallQual`) et la surface habitable (`GrLivArea`) ?*

**Documentation utile**

- [`pandas.Series.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_values.html)
- [`pandas.Series.plot.barh`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.barh.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `pandas.Series(rfr.feature_importances_, index=train_X.columns)` : une importance par variable, sous son nom ; leur somme vaut 1.
- `sort_values(ascending=False)` les range de la plus importante à la moins importante, et `.head(20)` garde les 20 premières.
- `[::-1]` inverse l'ordre, pour que la plus importante soit en haut du graphique en barres horizontales.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
importances = pandas.Series(..., index=train_X.columns)
importances = importances.sort_values(...)
importances.head(...)[::-1].plot.barh(figsize=(8, 6))
plt.show()
```

### Solution

In [ ]:
importances = pandas.Series(rfr.feature_importances_, index=train_X.columns)
importances = importances.sort_values(ascending=False)
importances.head(20)[::-1].plot.barh(figsize=(8, 6))
plt.title("Importance par impureté des 20 premières variables")
plt.show()

## Le premier arbre de la forêt

*Affichez les trois premiers niveaux du premier arbre de la forêt (`rfr.estimators_[0]`) avec `sklearn.tree.plot_tree`. Ses premières séparations portent-elles sur les variables les plus importantes trouvées plus haut ? Pourquoi ?*

**Documentation utile**

- [`sklearn.tree.plot_tree`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `max_depth=2` : la racine et deux niveaux en dessous ; l'arbre entier, de centaines de feuilles, serait illisible.
- `feature_names=list(train_X.columns)` : le nom des variables dans chaque nœud, au lieu de leur numéro.
- `filled=True` colore les nœuds selon leur prédiction ; `plt.figure(figsize=(14, 6))` avant l'appel donne la place de lire.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
plt.figure(figsize=(14, 6))
sklearn.tree.plot_tree(
  ...,
  max_depth=...,
  feature_names=...,
  filled=True,
  fontsize=10,
)
plt.show()
print(list(importances.index[:5]))
```

### Solution

In [ ]:
plt.figure(figsize=(14, 6))
sklearn.tree.plot_tree(
  rfr.estimators_[0],
  max_depth=2,
  feature_names=list(train_X.columns),
  filled=True,
  fontsize=10,
)
plt.show()
print("Les 5 variables les plus importantes :", list(importances.index[:5]))

Pas forcément. Dans la solution, la racine sépare sur `GarageArea`, cinquième variable par importance, et `OverallQual`, la plus importante, n'apparaît qu'au niveau suivant. À chaque séparation, l'arbre ne choisit que parmi un tiers des variables, tirées au hasard (`max_features=0.33`), et il est entraîné sur son propre échantillon de lignes : c'est ce qui décorrèle les arbres de la forêt. L'importance par impureté, moyennée sur tous les arbres, lisse ces hasards.

## Erreur out-of-bag (OOB)

Chaque arbre de la forêt est entraîné sur un tirage avec remise des lignes : environ 37 % des maisons (davantage si `max_samples` est inférieur à 1) n'y figurent pas, et sont *out-of-bag* pour cet arbre. L'erreur OOB d'une forêt prédit chaque maison avec les seuls arbres qui ne l'ont pas vue : c'est une validation gratuite, sans validation croisée (`oob_score=True`).

*Tracez l'erreur OOB (une RMSE) d'une copie de `rfr` en fonction de son nombre d'arbres, de 25 à 300 par pas de 25, avec la RMSE de validation croisée de la recherche en repère. À partir de combien d'arbres l'erreur se stabilise-t-elle ? Est-elle proche de la RMSE de validation croisée ?*

**Documentation utile**

- [`sklearn.base.clone`](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html)
- [`sklearn.metrics.root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)
- [`warm_start` dans scikit-learn](https://scikit-learn.org/stable/glossary.html#term-warm_start)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `sklearn.base.clone(rfr).set_params(warm_start=True, oob_score=True)` : une forêt non entraînée, de mêmes paramètres que `rfr`, qui reste intacte pour la suite.
- Avec `warm_start=True`, un `fit` après `set_params(n_estimators=n)` ajoute seulement les arbres qui manquent pour arriver à `n`, au lieu de réentraîner toute la forêt (comme dans la démonstration `random-forest-basics`).
- `forest.oob_prediction_` : la prédiction OOB de chaque maison de `train_X` ; `sklearn.metrics.root_mean_squared_error(train_y, forest.oob_prediction_)` en donne la RMSE. `forest.oob_score_` est, lui, le R² de ces prédictions, pas une erreur.
- `plt.axhline(-rscv.best_score_, linestyle="--")` trace le repère horizontal.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
forest = sklearn.base.clone(rfr).set_params(warm_start=..., oob_score=...)
n_trees = range(25, 301, 25)
oob_rmse = []
for n in n_trees:
  forest.set_params(n_estimators=n)
  forest.fit(...)
  oob_rmse.append(sklearn.metrics.root_mean_squared_error(train_y, ...))

plt.plot(n_trees, oob_rmse, label="Erreur OOB")
plt.axhline(..., color="gray", linestyle="--", label="Validation croisée")
plt.legend()
plt.show()
```

### Solution

In [ ]:
# Une copie non entraînée de rfr, qui reste intacte
forest = sklearn.base.clone(rfr).set_params(warm_start=True, oob_score=True)
n_trees = range(25, 301, 25)
oob_rmse = []
for n in n_trees:
  # warm_start=True : fit ajoute seulement les arbres qui manquent
  forest.set_params(n_estimators=n)
  forest.fit(train_X, train_y)
  oob_rmse.append(
    sklearn.metrics.root_mean_squared_error(train_y, forest.oob_prediction_)
  )

plt.plot(n_trees, oob_rmse, label="Erreur OOB")
plt.axhline(
  -rscv.best_score_,
  color="gray",
  linestyle="--",
  label="RMSE de validation croisée (recherche)",
)
plt.xlabel("Nombre d'arbres")
plt.ylabel("RMSE ($)")
plt.title("Erreur OOB selon le nombre d'arbres")
plt.legend()
plt.show()

L'erreur OOB baisse vite jusqu'à une centaine d'arbres, puis se stabilise vers 29 700 $ : au-delà de 100 à 150 arbres, en ajouter ne coûte plus que du temps. Elle reste un peu au-dessus de la RMSE de validation croisée (28 300 $), car chaque maison n'y est prédite que par les arbres qui ne l'ont pas tirée, environ la moitié de la forêt ici. Les deux estimations restent proches, et l'erreur OOB s'obtient en un seul entraînement, au lieu de 5.

## Évaluation sur le jeu de validation

Les RMSE précédentes ont servi à choisir le modèle : la meilleure des 20 configurations de la recherche est un peu optimiste. Le jeu de validation, mis de côté au début, n'a servi à aucun choix.

*Calculez la RMSE de `rfr` sur le jeu de validation (`val_X`, `val_y`), et comparez-la à la RMSE de validation croisée et à l'erreur OOB.*

**Documentation utile**

- [`sklearn.metrics.root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `rfr.predict(val_X)` : les prix prédits pour les maisons de validation.
- `sklearn.metrics.root_mean_squared_error(val_y, predictions)` : leur RMSE, en dollars.
- `-rscv.best_score_` et `oob_rmse[-1]` : les deux RMSE à comparer.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
val_rmse = sklearn.metrics.root_mean_squared_error(..., rfr.predict(...))
print(f"RMSE sur le jeu de validation : {val_rmse:.0f} $")
print(f"RMSE de validation croisée : {...:.0f} $")
print(f"Erreur OOB (300 arbres) : {...:.0f} $")
```

### Solution

In [ ]:
val_rmse = sklearn.metrics.root_mean_squared_error(val_y, rfr.predict(val_X))
print(f"RMSE sur le jeu de validation : {val_rmse:.0f} $")
print(f"RMSE de validation croisée : {-rscv.best_score_:.0f} $")
print(f"Erreur OOB (300 arbres) : {oob_rmse[-1]:.0f} $")

La RMSE sur le jeu de validation (environ 34 400 $) est plus haute que celle de la validation croisée (28 300 $), mais reste dans l'écart d'un pli à l'autre (de 20 000 $ à 36 000 $ selon le pli). Sur 292 maisons, quelques grosses erreurs pèsent lourd dans une RMSE : les deux plus grosses viennent ici de deux maisons de plus de 4 000 sq ft, celles que l'exploration propose de retirer de l'entraînement ; sans elles, la RMSE de validation tomberait vers 25 700 $. On ne les retire pas pour autant du jeu de validation : en production, on ne choisit pas les maisons à estimer. C'est cette estimation, sur des données qui n'ont servi à aucun choix, qu'on annonce pour le modèle.

## Conclusion

- Moyenner des arbres décorrélés réduit la variance d'un arbre seul : la forêt se trompe bien moins qu'un arbre, sans presque aucun réglage.
- Le réglage par recherche aléatoire n'apporte ensuite que peu (2 % ici).
- La forêt donne en plus une estimation de son erreur (OOB) et une importance des variables.
- Un jeu mis de côté dès le début, qui n'a servi à aucun choix, donne l'erreur à attendre sur de nouvelles maisons.

La cellule suivante compare la forêt réglée à des arbres boostés (gradient boosting), ici `HistGradientBoostingRegressor` de scikit-learn, sans aucun réglage : sur des données tabulaires, c'est aujourd'hui le modèle de référence, souvent un peu meilleur qu'une forêt.

In [ ]:
hgb_rmse = score(sklearn.ensemble.HistGradientBoostingRegressor(random_state=0))
print(f"HistGradientBoostingRegressor : {hgb_rmse[0]:.0f} $ ± {hgb_rmse[1]:.0f} $")
print(f"Forêt réglée : {-rscv.best_score_:.0f} $")